# Transformer Training for Sign Language Landmarks
This notebook trains a Transformer model on the pre-extracted facial/hand/pose landmarks.

In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## 1. Dataset Class
The landmark pipeline outputs `.npy` files containing temporal features.

In [2]:
class LandmarkDataset(Dataset):
    def __init__(self, data_dir, max_seq_len=15):
        self.data_dir = data_dir
        self.max_seq_len = max_seq_len
        self.file_paths = []
        self.labels = []
        
        # Get all subdirectories as classes
        self.classes = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(self.classes)
        
        for cls in self.classes:
            cls_dir = os.path.join(data_dir, cls)
            npy_files = glob.glob(os.path.join(cls_dir, "*.npy"))
            for f in npy_files:
                self.file_paths.append(f)
                self.labels.append(cls)
                
        self.encoded_labels = self.label_encoder.transform(self.labels)
        
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.encoded_labels[idx]
        
        # Load sequence of landmarks
        seq = np.load(file_path)
        seq = torch.tensor(seq, dtype=torch.float32)
        
        # Padding or truncating to max_seq_len
        if len(seq) > self.max_seq_len:
            seq = seq[:self.max_seq_len]
        elif len(seq) < self.max_seq_len:
            pad_size = self.max_seq_len - len(seq)
            seq = torch.cat([seq, torch.zeros(pad_size, seq.size(1))])
            
        return seq, torch.tensor(label, dtype=torch.long)

## 2. Transformer Model
We use a Transformer Encoder to process the sequence of landmarks and output a final classification.

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0)]
        return x

class SignTransformer(nn.Module):
    def __init__(self, input_dim=168, num_classes=10, d_model=128, nhead=8, num_layers=4, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Project input feature dimension to d_model
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc_out = nn.Linear(d_model, num_classes)
        
    def forward(self, src):
        # src shape: (batch_size, seq_len, input_dim)
        x = self.input_projection(src)
        
        # Swap batch and seq_len for Positional Encoding
        x = x.transpose(0, 1)  # (seq_len, batch_size, d_model)
        x = self.pos_encoder(x)
        x = x.transpose(0, 1)  # (batch_size, seq_len, d_model)
        
        output = self.transformer_encoder(x)
        
        # Aggregate over time (e.g., take the mean)
        output = torch.mean(output, dim=1)
        
        return self.fc_out(output)

## 3. Training Setup

In [4]:
# Hyperparameters
DATA_DIR = '../outputs/landmarks'
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 15 # Based on previous pipeline limits
INPUT_DIM = 168  # 56 landmarks * 3 coords
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = LandmarkDataset(data_dir=DATA_DIR, max_seq_len=MAX_SEQ_LEN)
num_classes = len(dataset.classes)
print(f"Total samples: {len(dataset)}, Classes: {num_classes}")

# Split Dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Initialize Model
model = SignTransformer(input_dim=INPUT_DIM, num_classes=num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print("Model Initialized on:", DEVICE)

Total samples: 187, Classes: 15
Model Initialized on: cuda


## 4. Training Loop

In [5]:
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(DEVICE), target.to(DEVICE)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
        
    train_acc = 100 * correct / total
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()
            
            _, predicted = torch.max(output.data, 1)
            val_total += target.size(0)
            val_correct += (predicted == target).sum().item()
            
    val_acc = 100 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)
    
    print(f'Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {total_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%')

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), '../checkpoints/transformer_best.pth')
        print("Saved new best model!")

Epoch [1/50] | Train Loss: 2.7877 | Train Acc: 9.40% | Val Loss: 2.7580 | Val Acc: 7.89%
Saved new best model!
Epoch [2/50] | Train Loss: 2.5784 | Train Acc: 17.45% | Val Loss: 2.6858 | Val Acc: 13.16%
Saved new best model!
Epoch [3/50] | Train Loss: 2.4387 | Train Acc: 27.52% | Val Loss: 2.6100 | Val Acc: 18.42%
Saved new best model!
Epoch [4/50] | Train Loss: 2.2858 | Train Acc: 35.57% | Val Loss: 2.5209 | Val Acc: 23.68%
Saved new best model!
Epoch [5/50] | Train Loss: 2.1558 | Train Acc: 38.26% | Val Loss: 2.3815 | Val Acc: 28.95%
Saved new best model!
Epoch [6/50] | Train Loss: 1.9803 | Train Acc: 40.94% | Val Loss: 2.3070 | Val Acc: 28.95%
Saved new best model!
Epoch [7/50] | Train Loss: 1.9008 | Train Acc: 51.01% | Val Loss: 2.2364 | Val Acc: 26.32%
Saved new best model!
Epoch [8/50] | Train Loss: 1.7903 | Train Acc: 52.35% | Val Loss: 2.2163 | Val Acc: 28.95%
Saved new best model!
Epoch [9/50] | Train Loss: 1.7208 | Train Acc: 55.70% | Val Loss: 2.1632 | Val Acc: 31.58%
Saved n